In [1]:
from datasets import load_from_disk
from gensim.corpora import Dictionary
import yaml

with open('./../../configs/notebooks/model02.yaml', 'r') as f:
  config = yaml.safe_load(f)

dataset = load_from_disk("../../data/processed/notebooks/ast")
code_dictionary = Dictionary.load('./../../data/processed/notebooks/ast/code_dictionary.pt')
docstring_dictionary = Dictionary.load('./../../data/processed/notebooks/ast/docstring_dictionary.pt')

code_dictionary.id2token = {
    v: k for k, v in code_dictionary.token2id.items()
}
docstring_dictionary.id2token = {
    v: k for k, v in docstring_dictionary.token2id.items()
}

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.hidden = None # final hidden state
        self.cell = None
        self.basic_rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, dropout=dropout, batch_first=True) # NLF

    def forward(self, X):
        embedded = self.embedding(X)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(embedded) # NLH, 1NH, 1NH
        return batch_first_output, (self.hidden, self.cell)

In [3]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, bos_id, eos_id, pad_id, dropout = 0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.hidden = None
        self.cell = None
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.pad_id = pad_id
        self.basic_rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, dropout=dropout, batch_first=True) # NLF
        self.output_layer = nn.Linear(self.hidden_dim, self.vocab_size)

    def init_hidden(self, encoder_states):
        self.hidden, self.cell = encoder_states

    def forward(self, X):
        # X is N, 1, F
        embedded = self.embedding(X)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(embedded, (self.hidden, self.cell))
        logits = self.output_layer(batch_first_output)
        return logits, (self.hidden, self.cell)

In [4]:
class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, teacher_forcing_prob=0.5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.teacher_forcing_prob = teacher_forcing_prob
        self.outputs = None

    def init_outputs(self, batch_size, target_len):
        device = next(self.parameters()).device
        # N, L, V (output is logits)
        self.outputs = torch.zeros(
            batch_size,
            target_len,
            self.decoder.vocab_size).to(device)

    def store_output(self, i, out):
        # Stores the output
        self.outputs[:, i:i+1, :] = out

    def forward(self, source_seq, target_seq):
      batch_size = source_seq.size(0)
      target_len = target_seq.size(1)

      _, (enc_hidden, enc_cell) = self.encoder(source_seq)

      self.decoder.init_hidden((enc_hidden, enc_cell))
      self.init_outputs(batch_size, target_len)

      dec_inputs = torch.full((batch_size, 1), 
                              self.decoder.bos_id,
                              dtype=torch.long).to(source_seq.device)
      
      for t in range(target_seq.size(1)):
        logits, _ = self.decoder(dec_inputs)
        self.outputs[:, t, :] = logits.squeeze(1)

        if self.training and torch.rand(1).item() < self.teacher_forcing_prob:
           dec_inputs = target_seq[:, t:t+1] # if teacher forcing, use actual next token as next input
        else:
           dec_inputs = logits.argmax(dim=-1) # else, use predicted token
        
      return self.outputs

In [5]:
from torch.nn.utils.rnn import pad_sequence
import torch

torch.cuda.empty_cache() # clears GPU memory
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [6]:
class CollateFn:
  def __init__(self, src_pad_id, tgt_pad_id):
    self.src_pad_id = src_pad_id
    self.tgt_pad_id = tgt_pad_id

  def __call__(self, batch):
    input_ids = [torch.tensor(x['input_ids'], dtype=torch.long) for x in batch]
    labels = []
    for x in batch:
      seq = x['labels']+[docstring_dictionary.token2id['[EOS]']]
      labels.append(torch.tensor(seq, dtype=torch.long))

    input_ids = pad_sequence(
      input_ids,
      batch_first=True,
      padding_value=self.src_pad_id
    )

    labels = pad_sequence(
      labels,
      batch_first=True,
      padding_value=self.tgt_pad_id
    )

    return input_ids, labels
  
collate = CollateFn(
  src_pad_id=code_dictionary.token2id['[PAD]'],
  tgt_pad_id=docstring_dictionary.token2id['[PAD]']
)

In [7]:
import os

def save_checkpoint(epoch, model, optimizer, val_loss):
    os.makedirs(config['checkpoint_dir'], exist_ok=True)
    checkpoint_path = f'{config["checkpoint_dir"]}/best_model.pt'
    torch.save({'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                }, checkpoint_path)
  
def load_checkpoint(model, optimizer):
    checkpoint_path = f'{config["checkpoint_dir"]}/best_model.pt'
    if(not os.path.exists(checkpoint_path)):
        print("No checkpoint found, starting from scratch")
        return 0
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"Loaded checkpoint from epoch {start_epoch}")

    for state in optimizer.state.values():
      for k, v in state.items():
          if isinstance(v, torch.Tensor):
              state[k] = v.to(device)
    return start_epoch

In [8]:
from torch.utils.data import DataLoader
generator = torch.Generator()
generator.manual_seed(42)
torch.manual_seed(42)

train_dataset = dataset['train'].select(range(config['train_data_length']))
valid_dataset = dataset['valid'].select(range(config['valid_data_length']))

batch_size = config['batch_size']

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

code_vocab_size = len(code_dictionary.token2id)
docstring_vocab_size = len(docstring_dictionary.token2id)
embedding_dim = config['embedding_dim']
hidden_dim = config['hidden_dim']

encoder = Encoder(
  vocab_size=code_vocab_size,
  embedding_dim=embedding_dim,
  hidden_dim=hidden_dim
)

decoder = Decoder(
  vocab_size=docstring_vocab_size, 
  embedding_dim=embedding_dim, 
  hidden_dim=hidden_dim,
  bos_id=docstring_dictionary.token2id['[BOS]'],
  eos_id=docstring_dictionary.token2id['[EOS]'],
  pad_id=docstring_dictionary.token2id['[PAD]']
)

model = EncoderDecoder(
  encoder=encoder,
  decoder=decoder,
  teacher_forcing_prob=config['teacher_forcing_prob']
)



loss = nn.CrossEntropyLoss(ignore_index=docstring_dictionary.token2id['[PAD]'])
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [9]:
epochs = config['num_epochs']
start_epoch = load_checkpoint(model, optimizer)+1

model.to(device)
best_loss = float('inf')
counter = 0

for epoch in range(start_epoch, epochs):
  model.train()
  batch_losses = []
  for input, labels in train_dataloader:
    input = input.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    optimizer.zero_grad()
    y_pred = model(input, labels)
    y_pred = y_pred.permute(0, 2, 1)  # N, V, L
    single_loss = loss(y_pred, labels)
    single_loss.backward()
    optimizer.step()
    batch_losses.append(single_loss.item())
  
  if(epoch % 5 == 0):
    print(f"Epoch {epoch:3}, Training Loss: {sum(batch_losses)/len(batch_losses):10.8f}")

  model.eval()
  with torch.no_grad():
    val_losses = []
    for input, labels in valid_dataloader:
      input = input.to('cuda' if torch.cuda.is_available() else 'cpu', non_blocking=True)
      labels = labels.to('cuda' if torch.cuda.is_available() else 'cpu', non_blocking=True)
      y_pred = model(input, labels)
      y_pred = y_pred.permute(0, 2, 1)  # N, V, L
      single_loss = loss(y_pred, labels)
      val_losses.append(single_loss.item())

  if epoch % 5 == 0:
    print(f"Epoch {epoch:3}, Validation Loss: {sum(val_losses)/len(val_losses):10.8f}")

  if(len(val_losses) > 0 and sum(val_losses)/len(val_losses) < best_loss):
    best_loss = sum(val_losses)/len(val_losses)
    print(f"Saving new best model epoch {epoch} with loss {best_loss:10.8f}")
    counter = 0
    save_checkpoint(epoch, model, optimizer, best_loss)
    
  else:
    counter += 1
    if counter >= config['patience']:
      print("Early stopping")
      break
  
  torch.cuda.empty_cache() # clears GPU memory
  del single_loss
  del y_pred
  del input
  del labels

No checkpoint found, starting from scratch
Saving new best model epoch 1 with loss 6.65283717
Epoch   5, Training Loss: 5.27774493
Epoch   5, Validation Loss: 7.12201467
Early stopping


In [10]:
def predict_ids(model, input_seq, max_length=50):
  model.eval()
  device = next(model.parameters()).device

  input_tensor = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)  # 1, L

  encoder_outputs, (enc_hidden, enc_cell) = model.encoder(input_tensor)
  model.decoder.init_hidden((enc_hidden, enc_cell))
  dec_input = torch.tensor([[model.decoder.bos_id]], dtype=torch.long).to(device)  # 1, 1

  pred_ids = []
  for _ in range(max_length):
    logits, (dec_hidden, dec_cell) = model.decoder(dec_input)

    temperature = 0.7
    probs = torch.softmax(logits / temperature, dim=-1)
    next_token_id = torch.multinomial(probs.squeeze(0), 1).item()

    if(next_token_id == model.decoder.eos_id):
      break
    pred_ids.append(next_token_id)
    dec_input = torch.tensor([[next_token_id]], dtype=torch.long).to(device)  # 1, 1
  
  return pred_ids
  


In [11]:
import ast

# Function to convert code string to AST
def code_to_ast(code_string):
  try:
    return ast.parse(code_string)
  except SyntaxError:
    return None # some code snippets are not valid (python 2 instead of python 3)

In [12]:
### Linearizing the AST (by passing first node) into a list of tokens
def linearize_ast(node, tokens):
  if node is None:
    return

  tokens.append(type(node).__name__)

  # specific node types
  # if it's a name we add the variable name
  if isinstance(node, ast.Name):
    tokens.append(f"VAR_{node.id}")

  # if it's a constant we add a placeholder to not write actual values
  elif isinstance(node, ast.Constant):
    tokens.append("CONST")

  # if it's a function argument we add the argument name
  elif isinstance(node, ast.arg):
    tokens.append(f"ARG_{node.arg}")

  for child in ast.iter_child_nodes(node):
    linearize_ast(child, tokens)

In [13]:
test_dataset = dataset['test'].select(range(config['test_data_length']))

test_dataloader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    pin_memory=True,
    collate_fn=collate
)

In [14]:
def decode_ids(pred_ids):
    return [docstring_dictionary.id2token[token_id] for token_id in pred_ids]

In [15]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

smooth_fn = SmoothingFunction().method4
bleu_scores = []
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
rouge_scores = []

test_dataset = dataset['test'].select(range(config['test_data_length']))

load_checkpoint(model, optimizer)
model.eval()
model.to(device)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    pin_memory=True,
    collate_fn=collate
)


for(input, labels) in test_dataloader:
  for i in range(input.size(0)):
    with torch.no_grad():
      pred_ids = predict_ids(model, input[i].tolist())
    pred_tokens  = decode_ids(pred_ids)
    labels_tokens = decode_ids(labels[i].tolist())

    print("Predicted Docstring: ", ' '.join(pred_tokens))
    print("Actual Docstring:    ", ' '.join(labels_tokens))

    bleu_score = sentence_bleu(
        [labels_tokens],
        pred_tokens,
        smoothing_function=smooth_fn
    )

    rouge_score = rouge.score(' '.join(labels_tokens), ' '.join(pred_tokens))['rougeL'].fmeasure

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)

print(f"Average BLEU score on test set: {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average RougeL score on test set: {sum(rouge_scores)/len(rouge_scores):.4f}")

Loaded checkpoint from epoch 1
Predicted Docstring:  Calculate the load the pylintrc the given scope .
Actual Docstring:     str - > list Convert XML to URL List . From [UNK] . [EOS]
Predicted Docstring:  Return a action to language info a . this . .
Actual Docstring:     Downloads [UNK] videos by URL . [EOS]
Predicted Docstring:  Convert the sequence . from the line . .
Actual Docstring:     Downloads [UNK] videos by URL . [EOS]
Predicted Docstring:  Process a avoiding .
Actual Docstring:     Format text with color or other effects into ANSI escaped string . [EOS]
Predicted Docstring:  Return the header .
Actual Docstring:     Print a log message to standard error . [EOS]
Predicted Docstring:  Get the given of the dictionary . bus
Actual Docstring:     Print an error log message . [EOS]
Predicted Docstring:  Process a HTML of is to .
Actual Docstring:     What a Terrible [UNK] [EOS]
Predicted Docstring:  Removes this .
Actual Docstring:     Detect operating system . [EOS]
Predicted Do

Now, we can see that the model does not behave that well. As we anticipated a couple of notebooks ago, we can definitely improve this encoder-decoder with some attention.